### 작업형 1 문제 구조  
작업형 1 문제의 출제 흐름과 정답 제출 방식을 익히기 위해 하나의 혼합형 데이터를 사용!  
같은 파일은 장 앞부분에서 한 번만 불러오고, 이후 항목에서는 df 데이터프레임을 이어서 사용

In [1]:
import pandas as pd

df = pd.read_csv('./data/part02_type1/type1_mixed_exam.csv')

print(df.shape)
print(df.head(3).to_string(index=False))

(600, 8)
case_id group  score  amount       date text_code  missing_value  flag
MX00001    G2  66.11   63500 2025-11-13   DD963-K           51.8     0
MX00002    G3  87.19  194300 2025-11-25   CC473-L           51.0     1
MX00003    G2  83.03  147700 2025-08-06   DD916-M           33.3     0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   case_id        600 non-null    object 
 1   group          600 non-null    object 
 2   score          600 non-null    float64
 3   amount         600 non-null    int64  
 4   date           600 non-null    object 
 5   text_code      600 non-null    object 
 6   missing_value  558 non-null    float64
 7   flag           600 non-null    int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 37.6+ KB


#### 작업형 1 출제 및 정답 제출 방식  
작업형 1 출제 방식  
작업형 1은 주어진 데이터를 읽고, 문제에서 요구한 조건에 맞게 값을 계산한 뒤, 최종 답 하나를 확인하는 유형이다.  
긴 모델링 과정이 필요한 작업형 2와 달리 판다스의 필터링, 집계, 정렬, 결측치 처리 등을 이용해 값을 정확히 찾는 능력을 본다.  
문제 문장은 보통 여러 지시를 짧게 이어서 제시한다. 따라서 코드를 바로 쓰기보다는 조건, 계산 대상, 답안 형태를 먼저 나누어 읽으면 풀이가 단순해진다.

!!!답안은 한 값으로 끝난다!!!  
작업형 1을 처음 보면 데이터 전체를 복잡하게 분석해야 할 것처럼 느껴질 수 있음. 그러나 대부분은 조건을 적용하고 특정 컬럼에서  
평균, 합계, 개수, 중앙값 같은 값을 계산해 답 하나를 얻는 흐름이다. 문제를 읽을 때는 '어떤 행을 남길지', '어떤 컬럼을 계산할지',  
'마지막에 어떤 형태로 출력할지'를 먼저 표시해야 한다!

1. 작업형 1에서 요구하는 것  
작업형 1의 기본 흐름은 다음 네 단계로 정리 가능!  

|대상 데이터|제시된 CSV 파일을 읽어 데이터프레임을 저장한다|  
|--|--|
|조건|조건식을 만들어 필요한 행만 선택함|  
|계산 대상|평균, 합계, 개수 등 계산할 칼럼을 고른다.|  
|답안 형태|정수, 소수점 자릿수, 문자열 등 요구 형식을 맞춘다.|

2. 문제 문장을 풀이 단계로 나누기  
예를 들어, 'flag가 1이고 group이 G2인 행에서 amount의 평균을 소수점 이하는 버리고 정수로 구하시오'라는 문장은 다음과 같이 나눌 수 있다.  

|조건1| flag가 1인 행|  
|--|--|  
|조건 2| group이 G2인 행|  
|계산 대상|amount의 평균|  
|답안 형태|소수점 이하를 버린 정수|

In [3]:
cond = (df['flag'] == 1) & (df['group'] == 'G2')  
ans = int(df.loc[cond, 'amount'].mean())
print(ans)

115865


In [4]:
# 더 직관적으로 코드 작성하는 법  
filtered = df[(df['flag'] == 1) & (df['group'] == 'G2')]
mean_amount = filtered['amount'].mean()
ans = int(mean_amount)
print(ans)

115865


즉, 단계형 문제에서는 앞 단계의 결과를 이어서 사용해야 함!  
문제가 '조건을 적용한 뒤 결측치를 채우고 평균을 구하시오'처럼 이어지며 각 단계가 독립된 문제가 아니라 하나의 흐름이라고 생각함.  
filtered, result 같은 새 변수를 사용하면 원본 데이터를 보존하면서 앞 단계의 결과를 다음 계산에 이어서 사용 가능

데이터 구조에 따른 풀이 순서  
|문제 유형|요구 사항|방법|  
|--|--|--|  
|그룹별 합계, 평균이 가장 큰 그룹| 집계 결과의 그룹명이 필요한다|groupby -> 집계 -> idxmax()|  
|각 그룹에서 값이 가장 큰 행|원본 행의 다른 컬럼이 필요한가|groupby()->idxmax()->loc[]|  
|두 기준의 집계 결과 비교|범주별 값을 같은 행에서 비교해야 하는가|pivot_table or unstack()|  
|여러 열에서 각각 가장 큰 행|열마다 세로 방향으로 비교하는가|wide 형태 확인 -> idxmax(axis=0)|

#### 정답 제출 방식  
1. 답안 입력 방식  
작업형 1에서는 계산 결과를 확인한 뒤 요구한 값만 답안란에 입력한다.  
작업형 2에서는 예측 결과를 지정된 구조의 CSV 파일로 저장해 제출한다.  
2. 최종값만 분명하게 출력하기  
앞에서 계산한 최종값을 answer 변수에 저장하고 마지막에 print(answer)만 실행하면 답안으로 옮길 값을 분명하게 확인 가능  


In [5]:
# 여러 값 중 제출할 값 고르기: 문제에서 그룹명을 묻는지 해당 그룹의 빈도수를 묻는지 잘 확인해야 함
group_counts = df['group'].value_counts()
print(group_counts)

group
G2    165
G1    158
G4    139
G3    138
Name: count, dtype: int64


In [6]:
print(group_counts.index[0])
print(group_counts.iloc[0])

G2
165


#### 정수 답안 처리와 출력값 확인  
1. 정수 답안 처리  
양수에서 소수점 이하를 제거하라고 하면 int(), 반올림하라고 하면 round()를 사용한다  

|문제 예시|요구|코드|  
|--|--|--|  
|소수점 이하는 버리시오|양수에서는 소수점 이하를 제거|int(answer)|  
|반올림하여 정수로 구하시오|가까운 정수로 반올림|round(answer)|  
|소수점 둘째 자리까지 구하시오|둘째 자리까지 반올림한다|round(answer, 2)|

In [7]:
# int로 소수점 이하 제거하기  
# 앞에서 사용한 조건에 맞는 amount 평균을 원래 값과 버리 처리한 값으로 비교
cond = ((df['flag'] == 1) & (df['group'] == 'G2'))
mean_amount = df.loc[cond, 'amount'].mean()  

print(mean_amount)
print(int(mean_amount))

115865.75342465754
115865


In [8]:
ans = round(mean_amount) # round의 경우에는 반올림되어 나타남
print(ans)

115866


In [9]:
# 처음부터 정수로 계산되는 값은 당연하지만 굳이 int가 필요 없음  
print((df['amount'] >= 150000).sum())
print(df['missing_value'].isna().sum())
print(df['flag'].sum())

140
42
278


#### 출력값 확인  
작업형 1에서는 코드를 잘 작성하는 것만큼 화면에 출력된 값이 실제 답안인지 확인하는 습관이 중요하다.  
풀이 과정에서는 여러 값을 출력할 수 있지만 답안란에 옮길 값은 문제 조건을 모두 반영한 최종 결과다.

In [10]:
# 중간 계산값 확인하기  
cond = df['group'] == 'G1'

print(cond.sum())
print(df.loc[cond, 'score'].mean())

158
65.08120253164557


In [11]:
cond = df['group'] == 'G1'
ans = round(df.loc[cond, 'score'].mean(), 2)

print(ans)

65.08


In [ ]:
# 여러 줄 출력에서 답안 찾기
print('flag가 1인 행 수', (df['flag'] == 1).sum())
print('missing value 결측치 수', df['missing_value'].isna().sum())

flag가 1인 행 수 278
missing value 결측치 수 42


: 